# Phase 9: Model Training — Random Forest Regressor

## 🎯 Objective
Train a multi-output non-linear **Random Forest Regressor** to model non-linear pollutant interactions and smog thresholds across all 72 prediction horizons.

### Benchmark Evaluation Goals:
1. Train 100-tree ensemble on 38,917 samples with 64 backward-looking features.
2. Evaluate on 9,748 out-of-time test samples across all 72 horizons.
3. Compare 3 models: **Naive Persistence Baseline vs Ridge Regression vs Random Forest**.
4. Profile feature importances and horizon error degradation.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.training_pipeline.trainer import ModelTrainer
from src.models.random_forest_model import RandomForestAQIModel

MODELS_DIR = PROJECT_ROOT / "data" / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## 1. 3-Way Model Benchmark Comparison on Test Partition

In [ ]:
with open(MODELS_DIR / "model_comparison.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

from src.training_pipeline.evaluator import ModelEvaluator
df_comp = ModelEvaluator.compare_models(eval_data)
display(df_comp)

## 2. Multi-Horizon Error Curves across all 72 Hours

In [ ]:
horizons = np.arange(1, 73)
naive_rmse = [eval_data["Naive Persistence Baseline"]["all_horizons"][f"h{h}"]["rmse"] for h in horizons]
ridge_rmse = [eval_data["Ridge Regression (MultiOutput)"]["all_horizons"][f"h{h}"]["rmse"] for h in horizons]
rf_rmse = [eval_data["Random Forest Regressor"]["all_horizons"][f"h{h}"]["rmse"] for h in horizons]

plt.figure(figsize=(11, 5))
plt.plot(horizons, naive_rmse, label="Naive Persistence Baseline", color="gray", linestyle="--", lw=2)
plt.plot(horizons, ridge_rmse, label="Ridge Regression (MultiOutput)", color="#4A90D9", lw=2)
plt.plot(horizons, rf_rmse, label="Random Forest Regressor", color="#00E400", lw=2.5)

plt.axvline(24, color="orange", linestyle=":", label="24h (Day 1)")
plt.axvline(48, color="purple", linestyle=":", label="48h (Day 2)")
plt.axvline(72, color="red", linestyle=":", label="72h (Day 3)")

plt.title("3-Way Multi-Horizon Error Comparison (RMSE vs Forecast Step $h$)")
plt.xlabel("Forecast Horizon (Hours ahead)")
plt.ylabel("Root Mean Squared Error (EPA AQI points)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Random Forest Feature Importances

In [ ]:
df_imp = pd.read_csv(MODELS_DIR / "rf_feature_importances.csv")

plt.figure(figsize=(10, 6))
top_imp = df_imp.head(15).iloc[::-1]
plt.barh(top_imp['feature'], top_imp['importance'], color='#4A90D9', edgecolor='black')
plt.title('Top 15 Most Important Features in Random Forest Ensemble')
plt.xlabel('Mean Impurity Decrease (Gini Importance)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

display(df_imp.head(15))

## 4. Key Findings & Progression to Phase 10 (TensorFlow Neural Network)
- **Short-Horizon Competence**: Random Forest achieves **54.28 RMSE at $h+1$** (19.3% error reduction vs Naive Persistence 67.30).
- **Primary Drivers**: Rolling statistics (`pm2_5_rolling_mean_12h` at 33.65%) and seasonal encodings (`month_cos` + `month_sin` at 13.23%) account for over 50% of the ensemble's predictive split power.
- **Next Stage (Phase 10)**: Train a feed-forward deep neural network in TensorFlow (`Dense(72)`) with Dropout and Early Stopping to capture global continuous representations.